In [0]:
import pandas as pd
import datetime as dt
df = spark.read.csv("/Volumes/workspace/default/ev_data/ev_sessions_raw.csv", header=True, inferSchema=True)

#display(df.limit(10))


def check_no_duplication_session(df):
    dupes = df.count() - df.select("session_id").distinct().count()
    return{"rule": "unique session_id", "failed_rows": dupes, "passed":dupes == 0}

def check_no_future_date(df,as_of_date):
    future = df.filter(df.session_date > as_of_date).count()
    return{"rule": "no future date", "failed_rows": future, "passed":future == 0}

def check_positive_revenue(df):
    neg_rev = df.filter(df.revenue_gbp < 0).count()
    return{"rule": "no negative revenue", "failed_rows": neg_rev, "passed":neg_rev == 0}

def check_vaild_status(df,accpet=['completed','failed','interrupted']): 
    valid_status = df.filter(~df.charger_status.isin(accpet)).count()
    return{"rule": "vaild status","failed_rows": valid_status, "passed":valid_status == 0} 

def check_valid_status(df): 
    valid_status = df.filter(~df.charger_status.isin(['completed', 'failed', 'interrupted'])).count()
    return {"rule": "valid status", "failed_rows": valid_status, "passed": valid_status == 0}

def check_no_null_critical(df):
    nulls = df.filter(df.session_date.isNull() | df.session_id.isNull() | df.kwh_delivered.isNull()).count()
    return {"rule": "no nulls", "failed_rows": nulls, "passed": nulls == 0}

print(check_no_null_critical(df))


{'rule': 'no nulls', 'failed_rows': 295, 'passed': False}
